In [2]:
import re
from src.utils import merge_data

In [3]:
raw = merge_data.data_combine().dropna().drop_duplicates()

In [4]:
raw.to_csv("raw.csv", encoding='utf-8-sig')

#### Xử lý Metadata & Spam:

Loại bỏ các đoạn text thừa của hệ thống VOZ như Sent from..., Quote:, Edit:.




In [5]:
infraction_keywords = [
    r"kích động", r"gây war", r"war", r"gây gổ", r"chửi", r"xúc phạm", r"thô tục",
    r"vô văn hóa", r"thiếu văn hóa", r"cãi nhau", r"thái độ", r"công kích",
    r"phân biệt", r"vùng miền", r"pbvm", r"kỳ thị", r"phản động", r"tổ lái", r"lái",
    r"chính trị", r"tôn giáo", r"sex", r"18\+", r"đồi trụy", r"nhạy cảm",
    r"bàn luận", r"lạc đề", r"spam", r"quảng cáo", r"seeding", r"nâng bi",
    r"dìm hàng", r"báo cũ", r"nguồn cấm", r"f33", r"f17", r"f\d+",
    r"thread", r"thớt", r"post", r"bài viết", r"comment", r"đào mộ", r"up bài",
    r"tiêu đề", r"tít", r"title", r"caps", r"viết hoa", r"không dấu",
    r"giá", r"sđt", r"điện thoại", r"địa chỉ", r"liên hệ", r"thông tin",
    r"lập lờ", r"gom", r"chung",
    r"vi phạm", r"rule", r"nội quy", r"quy định", r"k phù hợp", r"không phù hợp", r"ban",
    r"banned", r"xử lý", r"nhắc nhở", r"warning", r"clone"
]

infraction_regex = "|".join(infraction_keywords)

admin_junk_patterns = [
    r"(?:URL\s+)?nick bị (?:xử lý|ban|khóa|ra đảo):",
    r"URL thread/post bị (?:xử lý|ban|khóa|xóa):",
    r"Nick bị band (?:)",
    r"Mod\s+(?:xóa|ban|xử lý|nhắc nhở|warn|gộp|edit|chuyển)\s*:",
    fr"Lý do:.*(?:{infraction_regex})",
    r"Lý do bị band",
    r"Thời hạn:.*(?:vĩnh viễn|đến cuối năm|forever|\d+|[\d/\.-]+)",
    r"Thắc mắc:.*(?:tại sao|ban|nick|xóa|mod|admin|lý do)",
    r"voz không khuyến khích",
    r"chức năng report",
    r"vui lòng đọc kỹ nội quy",
    r"góp ý về việc",
    r"kiện cáo",
    r"https://voz.vn",
    r"^\s*@[a-z0-9]+\s*$",
    r"\s*via\s+thenextvoz[\s\S]*",
    r"(?:sent from|gửi từ).*(?:iphone|ipad|samsung|android|bphone|xiaomi|redmi|vsmart|pixel|blackberry|nokia|sony)[\s\S]*",
    r"(?:sent from|gửi từ).+using\s+(?:vozFApp|tapatalk|nextvoz)[\s\S]*",
    r"sent from my phone[\s\S]*",
    r"gửi từ điện thoại[\s\S]*",
    r"More options.*"

]

junk_regex = "|".join(admin_junk_patterns)

df_metadata = raw[~raw['text'].str.contains(junk_regex, case=False, na=False, regex=True)]
df_metadata['text'] = df_metadata['text'].str.strip()


In [6]:
df_metadata

,text
3,Bạn biết mình vi phạm cụ thể nội quy nào không...
8,"Bạn bị xử lý vì sử dụng ngôn từ không phù hợp,..."
10,1. Chỉ post link báo kèm quan điểm cá nhân của...
13,"Cám ơn Mod, cơ mà sao còm men về Ronaldo lại b..."
16,"Mod cho hỏi vì sao lại xoá thread này, bài báo..."
...,...
400615,muộn :(((
400616,khà khà giọng cũ đã quay trở lại
400617,Sớm luôn
400618,SỚM GẦN NHẤT


#### Xử lý Link & HTML:

Thay thế URL/Link, mention bằng token đặc biệt.




In [7]:
def replace_special_tokens(text):
    if not isinstance(text, str):
        return ""

    url_pattern = r"(?:http|https|ftp)://\S+|www\.\S+"
    text = re.sub(url_pattern, " <URL> ", text)

    email_pattern = r"\b[\w\.-]+@[\w\.-]+\.\w{2,}\b"
    text = re.sub(email_pattern, " <EMAIL> ", text)

    mention_pattern = r"@[\w_]+"
    text = re.sub(mention_pattern, " <USER> ", text)

    timestamp_pattern = r"\b\d{1,2}:\d{2}(?::\d{2})?\b"
    text = re.sub(timestamp_pattern, " <TIME> ", text)

    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_metadata['text'] = df_metadata['text'].apply(replace_special_tokens)

In [8]:
df_metadata.drop_duplicates().dropna().to_csv("special_token.csv",encoding="utf-8-sig")

In [11]:
english_filted.dropna().drop_duplicates().to_csv("english.csv", encoding='utf-8-sig')

#### Chuẩn hóa Unicode (Unicode Normalization):

Đưa toàn bộ về chuẩn Unicode dựng sẵn (NFC) (Quy tắc đặt thanh dấu kiểu cũ).

In [15]:
from src.Preprocess2.Normalizers.vietnamese_typing_normalizer import VietnameseNormalizer

vn = VietnameseNormalizer()

df_vt = english_filted['text'].apply(vn.normalize)



In [28]:
df_vn = df_vt.dropna().drop_duplicates()

In [29]:
df_vn

3         Bạn biết mình vi phạm cụ thể nội quy nào không...
8         Bạn bị xử lý vì sử dụng ngôn từ không phù hợp,...
10        1. Chỉ post link báo kèm quan điểm cá nhân của...
13        Cám ơn Mod, cơ mà sao còm men về Ronaldo lại b...
16        Mod cho hoỉ vì sao lại xoá thread này, bài báo...
                                ...                        
400615                                            muộn :(((
400616                     khà khà giọng cũ đã quay trở lại
400617                                             Sớm luôn
400618                                         SỚM GẦN NHẤT
400619                                  Sớm nhất luôn nghen
Name: text, Length: 259005, dtype: str

In [30]:
df_vn.to_csv("vn.csv", encoding='utf-8-sig')